In [ ]:
# Step 1: Import necessary libraries
import os
import cv2
import numpy as np
import git
from glob import glob
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Step 2: Clone the SAR Ship Detection GitHub repository (if not already cloned)
repo_url = 'https://github.com/Akashkalasagond/SAR-Ship-Detection.git'
repo_dir = '/content/SAR-Ship-Detection'
if not os.path.exists(repo_dir):
    git.Repo.clone_from(repo_url, repo_dir)

# Step 3: Define paths to images, labels, and mask directories
image_dir_train = '/content/SAR-Ship-Detection/train/images'
image_dir_test = '/content/SAR-Ship-Detection/test/images'
label_dir_train = '/content/SAR-Ship-Detection/train/labels'
label_dir_test = '/content/SAR-Ship-Detection/test/labels'
mask_dir_train = '/content/SAR-Ship-Detection/mask/train'
mask_dir_test = '/content/SAR-Ship-Detection/mask/test'

# Step 4: Ensure the mask directories exist
os.makedirs(mask_dir_train, exist_ok=True)
os.makedirs(mask_dir_test, exist_ok=True)

# Step 5: Generate masks for the dataset (train and test images)
def create_mask_for_dataset(image_dir, label_dir, mask_dir):
    image_paths = sorted(glob(os.path.join(image_dir, '*')))
    for img_path in tqdm(image_paths, desc='Generating Masks'):
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        label_path = os.path.join(label_dir, os.path.basename(img_path).replace('.jpg', '.txt'))
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    _, x, y, w_box, h_box = map(float, line.strip().split())
                    x1 = int((x - w_box/2) * w)
                    y1 = int((y - h_box/2) * h)
                    x2 = int((x + w_box/2) * w)
                    y2 = int((y + h_box/2) * h)
                    cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
        mask_path = os.path.join(mask_dir, os.path.basename(img_path).replace('.jpg', '.png'))
        cv2.imwrite(mask_path, mask)

# Generate masks for train and test images
tqdm.write("Generating masks for train images...")
create_mask_for_dataset(image_dir_train, label_dir_train, mask_dir_train)
tqdm.write("Generating masks for test images...")
create_mask_for_dataset(image_dir_test, label_dir_test, mask_dir_test)




Generating masks for train images...


Generating Masks: 100%|██████████| 3642/3642 [00:28<00:00, 129.69it/s]


Generating masks for test images...


Generating Masks: 100%|██████████| 1962/1962 [00:15<00:00, 128.54it/s]


In [ ]:
# Step 1: Import necessary libraries
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import Sequence
from glob import glob
from tqdm import tqdm

# Step 2: Limit GPU Memory Usage (Prevents Colab Crashes)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# Step 3: Define dataset paths
image_dir_train = '/content/SAR-Ship-Detection/train/images'
image_dir_test = '/content/SAR-Ship-Detection/test/images'
mask_dir_train = '/content/SAR-Ship-Detection/mask/train'
mask_dir_test = '/content/SAR-Ship-Detection/mask/test'

# Step 4: Load and preprocess dataset using a custom generator
class DataGenerator(Sequence):
    def __init__(self, image_paths, mask_paths, batch_size=8, target_size=(256, 256), **kwargs):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.target_size = target_size
        self.indexes = np.arange(len(self.image_paths))
        super().__init__(**kwargs)  # ✅ Fixes the UserWarning

    def __len__(self):
        return len(self.image_paths) // self.batch_size

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)  # Shuffle data each epoch

    def __getitem__(self, idx):
        batch_indexes = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        images, masks = [], []
        for i in batch_indexes:
            img = cv2.imread(self.image_paths[i])
            mask = cv2.imread(self.mask_paths[i], cv2.IMREAD_GRAYSCALE)
            if img is None or mask is None:
                continue
            img = cv2.resize(img, self.target_size) / 255.0
            mask = cv2.resize(mask, self.target_size) / 255.0
            mask = np.expand_dims(mask, axis=-1)  # Add channel dimension
            images.append(img)
            masks.append(mask)
        return np.array(images), np.array(masks)

# Load dataset file paths
train_img_paths = sorted(glob(os.path.join(image_dir_train, '*')))
train_mask_paths = sorted(glob(os.path.join(mask_dir_train, '*')))
test_img_paths = sorted(glob(os.path.join(image_dir_test, '*')))
test_mask_paths = sorted(glob(os.path.join(mask_dir_test, '*')))

train_gen = DataGenerator(train_img_paths, train_mask_paths, batch_size=8)
test_gen = DataGenerator(test_img_paths, test_mask_paths, batch_size=8)

# Step 5: Define an improved U-Net model
def unet_model(input_size=(256, 256, 3)):
    inputs = layers.Input(input_size)

    def conv_block(x, filters):
        x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        return x

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = conv_block(p3, 256)  # Bottleneck

    u1 = layers.UpSampling2D((2, 2))(c4)
    u1 = layers.concatenate([u1, c3])
    c5 = conv_block(u1, 128)

    u2 = layers.UpSampling2D((2, 2))(c5)
    u2 = layers.concatenate([u2, c2])
    c6 = conv_block(u2, 64)

    u3 = layers.UpSampling2D((2, 2))(c6)
    u3 = layers.concatenate([u3, c1])
    c7 = conv_block(u3, 32)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c7)

    model = models.Model(inputs, outputs)

    def dice_loss(y_true, y_pred):
        smooth = 1.0
        y_true_f = tf.keras.backend.flatten(y_true)
        y_pred_f = tf.keras.backend.flatten(y_pred)
        intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
        return 1 - ((2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth))

    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss=dice_loss, metrics=['accuracy'])
    return model

# Step 6: Train the model efficiently
model = unet_model()
callbacks_list = [
    callbacks.ModelCheckpoint('unet_sar_ship_best.keras', save_best_only=True),  # ✅ Fixed
    callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

history = model.fit(train_gen, validation_data=test_gen, epochs=25, callbacks=callbacks_list)

# Step 7: Save final model correctly
model.save('unet_sar_ship_final.keras')  # ✅ Fixed saving format

# Step 8: Evaluate the model
test_loss, test_accuracy = model.evaluate(test_gen)
print(f"Test Accuracy: {test_accuracy:.4f}")


Epoch 1/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 120s 191ms/step - accuracy: 0.7998 - loss: 0.9441 - val_accuracy: 0.9892 - val_loss: 0.7412
Epoch 2/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 82s 181ms/step - accuracy: 0.9272 - loss: 0.8949 - val_accuracy: 0.9424 - val_loss: 0.7422
Epoch 3/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 83s 182ms/step - accuracy: 0.9656 - loss: 0.8124 - val_accuracy: 0.9631 - val_loss: 0.6300
Epoch 4/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 83s 182ms/step - accuracy: 0.9844 - loss: 0.6583 - val_accuracy: 0.9935 - val_loss: 0.3117
Epoch 5/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 83s 182ms/step - accuracy: 0.9928 - loss: 0.4677 - val_accuracy: 0.9898 - val_loss: 0.3336
Epoch 6/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 83s 181ms/step - accuracy: 0.9939 - loss: 0.3557 - val_accuracy: 0.9878 - val_loss: 0.3359
Epoch 7/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 83s 182ms/step - accuracy: 0.9949 - loss: 0.2897 - val_accuracy: 0.9940 - val_loss: 0.2582
Epoch 8/25
455/455 ━━━━━━━━━━━━━━━━━━━━ 82s 179ms/step - accuracy: 0.9955 - loss: 

KeyboardInterrupt: 

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

# Paths
test_image_dir = "/content/SAR-Ship-Detection/test/images"
test_mask_dir = "/content/SAR-Ship-Detection/mask/test"
output_dir = "/content/SAR-Ship-Detection/detected_results"

# Create output directory if not exists
os.makedirs(output_dir, exist_ok=True)

# Load test images and masks
test_images = sorted(os.listdir(test_image_dir))
test_masks = sorted(os.listdir(test_mask_dir))

# Function to extract bounding boxes from masks
def get_bounding_boxes(mask):
    mask = (mask > 0.5).astype(np.uint8)  # Convert mask to binary
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = [cv2.boundingRect(cnt) for cnt in contours]  # Get bounding boxes
    return boxes

# Function to process and visualize ship detections
def detect_ships(image_name, mask_name):
    img_path = os.path.join(test_image_dir, image_name)
    mask_path = os.path.join(test_mask_dir, mask_name)

    # Load images and masks
    image = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Resize mask to match image size
    mask = cv2.resize(mask, (image.shape[1], image.shape[0]))

    # Get bounding boxes
    boxes = get_bounding_boxes(mask)

    # Draw bounding boxes on the image
    for (x, y, w, h) in boxes:
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Green box

    # Save the detected image
    save_path = os.path.join(output_dir, image_name)
    cv2.imwrite(save_path, image)

    return image, len(boxes)

# Process all test images and display results in batches
batch_size = 5  # Set how many images to display per batch
num_batches = len(test_images) // batch_size

for batch in range(num_batches + 1):
    fig, axes = plt.subplots(1, batch_size, figsize=(15, 5))

    for i in range(batch_size):
        index = batch * batch_size + i
        if index >= len(test_images):
            break  # Stop if out of images

        img_name, mask_name = test_images[index], test_masks[index]
        detected_img, ship_count = detect_ships(img_name, mask_name)

        # Display results
        axes[i].imshow(cv2.cvtColor(detected_img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f"Ships: {ship_count}")
        axes[i].axis("off")

    plt.show()


In [ ]:
import tensorflow.keras.backend as K

def iou_metric(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) - intersection + smooth)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def compute_metrics(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred > threshold).astype(int)
    y_true_bin = y_true.astype(int)

    precision = precision_score(y_true_bin.flatten(), y_pred_bin.flatten(), zero_division=1)
    recall = recall_score(y_true_bin.flatten(), y_pred_bin.flatten(), zero_division=1)
    f1 = f1_score(y_true_bin.flatten(), y_pred_bin.flatten(), zero_division=1)

    return precision, recall, f1


In [ ]:
import numpy as np

# Load the trained model
model = tf.keras.models.load_model('unet_sar_ship_best.keras', custom_objects={'dice_loss': iou_metric})

# Get predictions
y_true = []
y_pred = []

for i in range(len(test_gen)):
    X_batch, Y_batch = test_gen[i]
    Y_pred_batch = model.predict(X_batch)

    y_true.append(Y_batch)
    y_pred.append(Y_pred_batch)

# Convert lists to numpy arrays
y_true = np.vstack(y_true)
y_pred = np.vstack(y_pred)

# Compute IoU Score
iou_score = iou_metric(y_true, y_pred)
print(f"IoU Score: {K.eval(iou_score):.4f}")

# Compute Precision, Recall, and F1-Score
precision, recall, f1 = compute_metrics(y_true, y_pred)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Compute Final Accuracy
test_loss, test_accuracy = model.evaluate(test_gen)
print(f"Test Accuracy: {test_accuracy:.4f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━

InvalidArgumentError: cannot compute Mul as input #1(zero-based) was expected to be a double tensor but is a float tensor [Op:Mul] name: 

In [ ]:
import os
import cv2
import numpy as np
from glob import glob

# Confidence threshold for valid bounding boxes
CONF_THRESH = 0.3

# Define test directories
test_image_dir = "/content/SAR-Ship-Detection/test/images"
test_mask_dir = "/content/SAR-Ship-Detection/mask/test"
test_label_dir = "/content/SAR-Ship-Detection/test/labels"

# Load test dataset file paths
test_images = sorted(glob(os.path.join(test_image_dir, "*")))
test_masks = sorted(glob(os.path.join(test_mask_dir, "*")))
test_labels = sorted(glob(os.path.join(test_label_dir, "*")))

def get_bounding_boxes(mask):
    """ Extract bounding boxes from a binary mask using contour detection. """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h < 50:  # Ignore tiny detections
            continue
        boxes.append((x, y, w, h))
    return boxes

def compute_iou(boxA, boxB):
    """ Compute Intersection over Union (IoU) between two bounding boxes. """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]

    iou = interArea / (boxAArea + boxBArea - interArea)
    return iou

def non_maximum_suppression(boxes, iou_thresh=0.5):
    """ Apply NMS to remove redundant bounding boxes. """
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda x: x[2] * x[3], reverse=True)
    keep_boxes = []

    while boxes:
        best_box = boxes.pop(0)
        keep_boxes.append(best_box)
        boxes = [b for b in boxes if compute_iou(best_box, b) < iou_thresh]

    return keep_boxes

def calculate_metrics():
    tp, fp, fn = 0, 0, 0
    iou_scores = []

    for img_path, mask_path, label_path in zip(test_images, test_masks, test_labels):
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None or mask is None:
            print(f"⚠️ Skipping {img_path} (Image/Mask Not Found)")
            continue

        mask = cv2.resize(mask, (image.shape[1], image.shape[0]))

        # Morphological enhancement
        mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)

        predicted_boxes = get_bounding_boxes(mask)
        predicted_boxes = non_maximum_suppression(predicted_boxes, iou_thresh=0.5)

        gt_boxes = []
        if os.path.exists(label_path):
            with open(label_path, "r") as file:
                for line in file.readlines():
                    data = line.strip().split()
                    if len(data) < 5:
                        continue
                    _, x_center, y_center, w, h = map(float, data)
                    if w < 0.01 or h < 0.01:
                        continue

                    x = int((x_center - w / 2) * image.shape[1])
                    y = int((y_center - h / 2) * image.shape[0])
                    w = max(int(w * image.shape[1]), 1)
                    h = max(int(h * image.shape[0]), 1)

                    gt_boxes.append((x, y, w, h))

        matched = set()
        for p_box in predicted_boxes:
            best_iou = 0
            best_match = -1
            for i, t_box in enumerate(gt_boxes):
                if i in matched:
                    continue
                iou = compute_iou(p_box, t_box)
                if iou > best_iou:
                    best_iou = iou
                    best_match = i

            if not gt_boxes:
                fp += 1
                continue

            ship_area = p_box[2] * p_box[3]
            if ship_area > 600:
                dynamic_iou_thresh = 0.7
            elif ship_area > 200:
                dynamic_iou_thresh = 0.6
            else:
                dynamic_iou_thresh = 0.5

            if best_iou >= dynamic_iou_thresh:
                tp += 1
                matched.add(best_match)
                iou_scores.append(best_iou)
            else:
                fp += 1

        fn += len(gt_boxes) - len(matched)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0

    avg_iou = np.mean(iou_scores) if iou_scores else 0
    min_iou = np.min(iou_scores) if iou_scores else 0
    max_iou = np.max(iou_scores) if iou_scores else 0

    print(f"✅ Ship Detection Accuracy: {accuracy:.2f}%")
    print(f"✅ Ship Detection Precision: {precision:.4f}")
    print(f"✅ Ship Detection Recall: {recall:.4f}")
    print(f"✅ Ship Detection F1-Score: {f1_score:.4f}")
    print(f"✅ TP: {tp}, FP: {fp}, FN: {fn}")
    print(f"📊 Average IoU: {avg_iou:.4f}")
    print(f"📊 Minimum IoU: {min_iou:.4f}")
    print(f"📊 Maximum IoU: {max_iou:.4f}")

calculate_metrics()


✅ Ship Detection Accuracy: 85.46%
✅ Ship Detection Precision: 0.8731
✅ Ship Detection Recall: 0.8546
✅ Ship Detection F1-Score: 0.8638
✅ TP: 4701, FP: 683, FN: 800
📊 Average IoU: 0.8087
📊 Minimum IoU: 0.5357
📊 Maximum IoU: 0.9946


In [ ]:
import os
import cv2
import numpy as np

# Confidence threshold for valid bounding boxes
CONF_THRESH = 0.3

def get_bounding_boxes(mask):
    """ Extract bounding boxes from a binary mask using contour detection. """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h < 50:  # Ignore tiny detections
            continue
        boxes.append((x, y, w, h))
    return boxes

def compute_iou(boxA, boxB):
    """ Compute Intersection over Union (IoU) between two bounding boxes. """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]

    iou = interArea / (boxAArea + boxBArea - interArea)
    return iou

def non_maximum_suppression(boxes, iou_thresh=0.5):
    """ Apply NMS to remove redundant bounding boxes. """
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda x: x[2] * x[3], reverse=True)
    keep_boxes = []

    while boxes:
        best_box = boxes.pop(0)
        keep_boxes.append(best_box)
        boxes = [b for b in boxes if compute_iou(best_box, b) < iou_thresh]

    return keep_boxes

def calculate_metrics():
    tp, fp, fn = 0, 0, 0
    iou_scores = []

    for img_name, mask_name, label_name in zip(test_images, test_masks, test_labels):
        img_path = os.path.join(test_image_dir, img_name)
        mask_path = os.path.join(test_mask_dir, mask_name)
        label_path = os.path.join(test_label_dir, label_name)

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]))

        # Morphological enhancement
        mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)

        predicted_boxes = get_bounding_boxes(mask)
        predicted_boxes = non_maximum_suppression(predicted_boxes, iou_thresh=0.5)

        gt_boxes = []
        if os.path.exists(label_path):
            with open(label_path, "r") as file:
                for line in file.readlines():
                    data = line.strip().split()
                    if len(data) < 5:
                        continue
                    _, x_center, y_center, w, h = map(float, data)
                    if w < 0.01 or h < 0.01:
                        continue

                    x = int((x_center - w / 2) * image.shape[1])
                    y = int((y_center - h / 2) * image.shape[0])
                    w = max(int(w * image.shape[1]), 1)
                    h = max(int(h * image.shape[0]), 1)

                    gt_boxes.append((x, y, w, h))

        matched = set()
        for p_box in predicted_boxes:
            best_iou = 0
            best_match = -1
            for i, t_box in enumerate(gt_boxes):
                if i in matched:
                    continue
                iou = compute_iou(p_box, t_box)
                if iou > best_iou:
                    best_iou = iou
                    best_match = i

            if not gt_boxes:
                fp += 1
                continue

            ship_area = p_box[2] * p_box[3]
            if ship_area > 600:
                dynamic_iou_thresh = 0.7
            elif ship_area > 200:
                dynamic_iou_thresh = 0.6
            else:
                dynamic_iou_thresh = 0.5

            if best_iou >= dynamic_iou_thresh:
                tp += 1
                matched.add(best_match)
                iou_scores.append(best_iou)
            else:
                fp += 1

        fn += len(gt_boxes) - len(matched)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0

    avg_iou = np.mean(iou_scores) if iou_scores else 0
    min_iou = np.min(iou_scores) if iou_scores else 0
    max_iou = np.max(iou_scores) if iou_scores else 0

    print(f"✅ Ship Detection Accuracy: {accuracy:.2f}%")
    print(f"✅ Ship Detection Precision: {precision:.4f}")
    print(f"✅ Ship Detection Recall: {recall:.4f}")
    print(f"✅ Ship Detection F1-Score: {f1_score:.4f}")
    print(f"✅ TP: {tp}, FP: {fp}, FN: {fn}")
    print(f"📊 Average IoU: {avg_iou:.4f}")
    print(f"📊 Minimum IoU: {min_iou:.4f}")
    print(f"📊 Maximum IoU: {max_iou:.4f}")

calculate_metrics()


✅ Ship Detection Accuracy: 85.46%
✅ Ship Detection Precision: 0.8731
✅ Ship Detection Recall: 0.8546
✅ Ship Detection F1-Score: 0.8638
✅ TP: 4701, FP: 683, FN: 800
📊 Average IoU: 0.8087
📊 Minimum IoU: 0.5357
📊 Maximum IoU: 0.9946


In [ ]:
import os
import cv2
import numpy as np

CONF_THRESH = 0.3  # Keep confidence threshold the same

def get_bounding_boxes(mask):
    """Extract bounding boxes from a binary mask using contour detection."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h < 50:  # Restored original threshold (not 30)
            continue
        boxes.append((x, y, w, h))
    return boxes

def compute_iou(boxA, boxB):
    """Compute IoU between two bounding boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]

    iou = interArea / (boxAArea + boxBArea - interArea)
    return iou

def non_maximum_suppression(boxes, iou_thresh=0.5):
    """Apply NMS to remove redundant bounding boxes."""
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda x: x[2] * x[3], reverse=True)
    keep_boxes = []

    while boxes:
        best_box = boxes.pop(0)
        keep_boxes.append(best_box)
        boxes = [b for b in boxes if compute_iou(best_box, b) < iou_thresh]

    return keep_boxes

def calculate_metrics():
    tp, fp, fn = 0, 0, 0
    iou_scores = []

    for img_name, mask_name, label_name in zip(test_images, test_masks, test_labels):
        img_path = os.path.join(test_image_dir, img_name)
        mask_path = os.path.join(test_mask_dir, mask_name)
        label_path = os.path.join(test_label_dir, label_name)

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]))

        # ⚠ Fix: Remove over-dilation, use gentler processing
        kernel = np.ones((3, 3), np.uint8)  # Smaller kernel
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)  # Closing instead of dilation

        predicted_boxes = get_bounding_boxes(mask)
        predicted_boxes = non_maximum_suppression(predicted_boxes, iou_thresh=0.5)  # Reset to 0.5

        gt_boxes = []
        if os.path.exists(label_path):
            with open(label_path, "r") as file:
                for line in file.readlines():
                    data = line.strip().split()
                    if len(data) < 5:
                        continue
                    _, x_center, y_center, w, h = map(float, data)
                    if w < 0.01 or h < 0.01:
                        continue

                    x = int((x_center - w / 2) * image.shape[1])
                    y = int((y_center - h / 2) * image.shape[0])
                    w = max(int(w * image.shape[1]), 1)
                    h = max(int(h * image.shape[0]), 1)

                    gt_boxes.append((x, y, w, h))

        matched = set()
        for p_box in predicted_boxes:
            best_iou = 0
            best_match = -1
            for i, t_box in enumerate(gt_boxes):
                if i in matched:
                    continue
                iou = compute_iou(p_box, t_box)
                if iou > best_iou:
                    best_iou = iou
                    best_match = i

            if not gt_boxes:
                fp += 1
                continue

            # ⚠ Fix: Restore old IoU thresholds
            if best_iou >= 0.5:  # Changed from dynamic 0.75 to a simple 0.5
                tp += 1
                matched.add(best_match)
                iou_scores.append(best_iou)
            else:
                fp += 1

        fn += len(gt_boxes) - len(matched)

    # Compute Metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0

    avg_iou = np.mean(iou_scores) if iou_scores else 0
    min_iou = np.min(iou_scores) if iou_scores else 0
    max_iou = np.max(iou_scores) if iou_scores else 0

    # Print Results
    print(f"✅ Ship Detection Accuracy: {accuracy:.2f}%")
    print(f"✅ Ship Detection Precision: {precision:.4f}")
    print(f"✅ Ship Detection Recall: {recall:.4f}")
    print(f"✅ Ship Detection F1-Score: {f1_score:.4f}")
    print(f"✅ TP: {tp}, FP: {fp}, FN: {fn}")
    print(f"📊 Average IoU: {avg_iou:.4f}")
    print(f"📊 Minimum IoU: {min_iou:.4f}")
    print(f"📊 Maximum IoU: {max_iou:.4f}")

calculate_metrics()


✅ Ship Detection Accuracy: 88.15%
✅ Ship Detection Precision: 0.8990
✅ Ship Detection Recall: 0.8815
✅ Ship Detection F1-Score: 0.8901
✅ TP: 4849, FP: 545, FN: 652
📊 Average IoU: 0.9166
📊 Minimum IoU: 0.5014
📊 Maximum IoU: 1.0000


In [ ]:
import os
import cv2
import numpy as np

def get_bounding_boxes(mask):
    """Extract bounding boxes from the mask using contours."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h > 50:  # Ignore very small detections
            boxes.append((x, y, w, h))
    return boxes

def compute_iou(boxA, boxB):
    """Compute IoU between two bounding boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]

    iou = interArea / (boxAArea + boxBArea - interArea) if (boxAArea + boxBArea - interArea) > 0 else 0
    return iou

def non_maximum_suppression(boxes, iou_thresh=0.5):
    """Apply NMS to remove redundant detections."""
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda x: x[2] * x[3], reverse=True)
    keep_boxes = []

    while boxes:
        main_box = boxes.pop(0)
        keep_boxes.append(main_box)
        boxes = [b for b in boxes if compute_iou(main_box, b) < iou_thresh]

    return keep_boxes

# Define dataset paths
image_dir_test = '/content/SAR-Ship-Detection/test/images'
label_dir_test = '/content/SAR-Ship-Detection/test/labels'
mask_dir_test = '/content/SAR-Ship-Detection/mask/test'

# Validate directories
for path in [image_dir_test, label_dir_test, mask_dir_test]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ Directory does not exist: {path}")
    print(f"✅ Directory exists: {path}")

# Load test dataset files
test_images = sorted(os.listdir(image_dir_test))
test_masks = sorted(os.listdir(mask_dir_test))
test_labels = sorted(os.listdir(label_dir_test))

# Print sample filenames for debugging
print("Sample test images:", test_images[:5])
print("Sample test masks:", test_masks[:5])
print("Sample test labels:", test_labels[:5])

def calculate_metrics():
    tp, fp, fn = 0, 0, 0
    iou_scores = []

    for img_name, mask_name, label_name in zip(test_images, test_masks, test_labels):
        img_path = os.path.join(image_dir_test, img_name)
        mask_path = os.path.join(mask_dir_test, mask_name)
        label_path = os.path.join(label_dir_test, label_name)

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None or image is None:
            continue

        mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
        _, mask = cv2.threshold(mask, 128, 255, cv2.THRESH_BINARY)

        predicted_boxes = get_bounding_boxes(mask)
        predicted_boxes = non_maximum_suppression(predicted_boxes, iou_thresh=0.4)

        gt_boxes = []
        if os.path.exists(label_path):
            with open(label_path, "r") as file:
                for line in file.readlines():
                    data = line.strip().split()
                    if len(data) < 5:
                        continue
                    _, x_center, y_center, w, h = map(float, data)

                    if w < 0.01 or h < 0.01:
                        continue

                    x = int((x_center - w / 2) * image.shape[1])
                    y = int((y_center - h / 2) * image.shape[0])
                    w = max(int(w * image.shape[1]), 1)
                    h = max(int(h * image.shape[0]), 1)

                    gt_boxes.append((x, y, w, h))

        matched = set()
        for p_box in predicted_boxes:
            best_iou, best_match = 0, -1
            for i, t_box in enumerate(gt_boxes):
                if i in matched:
                    continue
                iou = compute_iou(p_box, t_box)
                if iou > best_iou:
                    best_iou, best_match = iou, i

            if not gt_boxes:
                fp += 1
                continue

            if best_iou >= 0.4:
                tp += 1
                matched.add(best_match)
                iou_scores.append(best_iou)
            else:
                fp += 1

        fn += len(gt_boxes) - len(matched)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0

    avg_iou = np.mean(iou_scores) if iou_scores else 0
    min_iou = np.min(iou_scores) if iou_scores else 0
    max_iou = np.max(iou_scores) if iou_scores else 0

    print(f"✅ Ship Detection Accuracy: {accuracy:.2f}%")
    print(f"✅ Ship Detection Precision: {precision:.4f}")
    print(f"✅ Ship Detection Recall: {recall:.4f}")
    print(f"✅ Ship Detection F1-Score: {f1_score:.4f}")
    print(f"✅ TP: {tp}, FP: {fp}, FN: {fn}")
    print(f"📊 Average IoU: {avg_iou:.4f}")
    print(f"📊 Minimum IoU: {min_iou:.4f}")
    print(f"📊 Maximum IoU: {max_iou:.4f}")

# Run the evaluation
calculate_metrics()

✅ Directory exists: /content/SAR-Ship-Detection/test/images
✅ Directory exists: /content/SAR-Ship-Detection/test/labels
✅ Directory exists: /content/SAR-Ship-Detection/mask/test
Sample test images: ['P0001_0_800_10190_10990.jpg', 'P0001_1200_2000_10190_10990.jpg', 'P0001_1200_2000_3600_4400.jpg', 'P0001_1800_2600_0_800.jpg', 'P0001_1800_2600_7800_8600.jpg']
Sample test masks: ['P0001_0_800_10190_10990.png', 'P0001_1200_2000_10190_10990.png', 'P0001_1200_2000_3600_4400.png', 'P0001_1800_2600_0_800.png', 'P0001_1800_2600_7800_8600.png']
Sample test labels: ['P0001_0_800_10190_10990.txt', 'P0001_1200_2000_10190_10990.txt', 'P0001_1200_2000_3600_4400.txt', 'P0001_1800_2600_0_800.txt', 'P0001_1800_2600_7800_8600.txt']
✅ Ship Detection Accuracy: 91.09%
✅ Ship Detection Precision: 0.9158
✅ Ship Detection Recall: 0.9109
✅ Ship Detection F1-Score: 0.9133
✅ TP: 5011, FP: 461, FN: 490
📊 Average IoU: 0.9110
📊 Minimum IoU: 0.4000
📊 Maximum IoU: 1.0000


In [ ]:
import os

# Define paths
image_dir_train = '/content/SAR-Ship-Detection/train/images'
image_dir_test = '/content/SAR-Ship-Detection/test/images'
label_dir_train = '/content/SAR-Ship-Detection/train/labels'
label_dir_test = '/content/SAR-Ship-Detection/test/labels'
mask_dir_train = '/content/SAR-Ship-Detection/mask/train'
mask_dir_test = '/content/SAR-Ship-Detection/mask/test'

# Check if directories exist
paths = [image_dir_train, image_dir_test, label_dir_train, label_dir_test, mask_dir_train, mask_dir_test]

for path in paths:
    if not os.path.exists(path):
        print(f"❌ Directory not found: {path}")
    else:
        print(f"✅ Directory exists: {path}")


✅ Directory exists: /content/SAR-Ship-Detection/train/images
✅ Directory exists: /content/SAR-Ship-Detection/test/images
✅ Directory exists: /content/SAR-Ship-Detection/train/labels
✅ Directory exists: /content/SAR-Ship-Detection/test/labels
✅ Directory exists: /content/SAR-Ship-Detection/mask/train
✅ Directory exists: /content/SAR-Ship-Detection/mask/test


In [ ]:
for path in paths:
    files = os.listdir(path)
    if len(files) == 0:
        print(f"⚠️ No files found in: {path}")
    else:
        print(f"✅ {len(files)} files found in: {path}")


✅ 3642 files found in: /content/SAR-Ship-Detection/train/images
✅ 1962 files found in: /content/SAR-Ship-Detection/test/images
✅ 3642 files found in: /content/SAR-Ship-Detection/train/labels
✅ 1962 files found in: /content/SAR-Ship-Detection/test/labels
✅ 3642 files found in: /content/SAR-Ship-Detection/mask/train
✅ 1962 files found in: /content/SAR-Ship-Detection/mask/test


In [ ]:
import os
import cv2
import numpy as np

# Paths to your dataset
image_dir_test = '/content/SAR-Ship-Detection/test/images'
label_dir_test = '/content/SAR-Ship-Detection/test/labels'
mask_dir_test = '/content/SAR-Ship-Detection/mask/test'

# IoU Calculation
def compute_iou(boxA, boxB):
    """Compute Intersection over Union (IoU) between two bounding boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]

    iou = interArea / (boxAArea + boxBArea - interArea) if (boxAArea + boxBArea - interArea) > 0 else 0
    return iou

# Extract bounding boxes from binary mask
def get_bounding_boxes(mask):
    """Extract bounding boxes from segmentation mask."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h > 50:  # Ignore small detections
            boxes.append((x, y, w, h))
    return boxes

# Non-Maximum Suppression
def non_maximum_suppression(boxes, iou_thresh=0.5):
    """Apply NMS to remove redundant detections."""
    if len(boxes) == 0:
        return []

    boxes = sorted(boxes, key=lambda x: x[2] * x[3], reverse=True)
    keep_boxes = []

    while boxes:
        main_box = boxes.pop(0)
        keep_boxes.append(main_box)
        boxes = [b for b in boxes if compute_iou(main_box, b) < iou_thresh]

    return keep_boxes

# Load test dataset
test_images = sorted(os.listdir(image_dir_test))
test_masks = sorted(os.listdir(mask_dir_test))
test_labels = sorted(os.listdir(label_dir_test))

# Evaluate Performance
def calculate_metrics():
    tp, fp, fn = 0, 0, 0
    iou_scores = []
    precisions, recalls = [], []
    iou_thresholds = np.arange(0.5, 1.0, 0.05)  # IoU thresholds for mAP@[0.5:0.95]

    all_precisions = {iou_thresh: [] for iou_thresh in iou_thresholds}

    for img_name, mask_name, label_name in zip(test_images, test_masks, test_labels):
        img_path = os.path.join(image_dir_test, img_name)
        mask_path = os.path.join(mask_dir_test, mask_name)
        label_path = os.path.join(label_dir_test, label_name)

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None or image is None:
            continue

        mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
        _, mask = cv2.threshold(mask, 128, 255, cv2.THRESH_BINARY)

        predicted_boxes = get_bounding_boxes(mask)
        predicted_boxes = non_maximum_suppression(predicted_boxes, iou_thresh=0.4)

        # Load ground truth labels
        gt_boxes = []
        if os.path.exists(label_path):
            with open(label_path, "r") as file:
                for line in file.readlines():
                    data = line.strip().split()
                    if len(data) < 5:
                        continue
                    _, x_center, y_center, w, h = map(float, data)

                    if w < 0.01 or h < 0.01:
                        continue

                    x = int((x_center - w / 2) * image.shape[1])
                    y = int((y_center - h / 2) * image.shape[0])
                    w = max(int(w * image.shape[1]), 1)
                    h = max(int(h * image.shape[0]), 1)

                    gt_boxes.append((x, y, w, h))

        matched = set()
        for p_box in predicted_boxes:
            best_iou = 0
            best_match = -1
            for i, t_box in enumerate(gt_boxes):
                if i in matched:
                    continue
                iou = compute_iou(p_box, t_box)
                if iou > best_iou:
                    best_iou = iou
                    best_match = i

            if not gt_boxes:
                fp += 1
                continue

            if best_iou >= 0.4:
                tp += 1
                matched.add(best_match)
                iou_scores.append(best_iou)
            else:
                fp += 1

        fn += len(gt_boxes) - len(matched)

        # Compute per-image precision-recall
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        precisions.append(precision)
        recalls.append(recall)

        for iou_thresh in iou_thresholds:
            tp_iou = sum(1 for i, t_box in enumerate(gt_boxes) if any(compute_iou(t_box, p_box) >= iou_thresh for p_box in predicted_boxes))
            fp_iou = len(predicted_boxes) - tp_iou
            fn_iou = len(gt_boxes) - tp_iou
            precision_iou = tp_iou / (tp_iou + fp_iou) if (tp_iou + fp_iou) > 0 else 0
            all_precisions[iou_thresh].append(precision_iou)

    # Compute final mAP values
    mAP_50 = np.mean(all_precisions[0.5]) if all_precisions[0.5] else 0
    mAP_50_95 = np.mean([np.mean(all_precisions[iou]) for iou in iou_thresholds]) if all_precisions else 0

    print(f"✅ Ship Detection Accuracy: {(tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0:.2f}%")
    print(f"✅ Ship Detection Precision: {np.mean(precisions) if precisions else 0:.4f}")
    print(f"✅ Ship Detection Recall: {np.mean(recalls) if recalls else 0:.4f}")
    print(f"✅ Ship Detection F1-Score: {(2 * np.mean(precisions) * np.mean(recalls)) / (np.mean(precisions) + np.mean(recalls)) if (np.mean(precisions) + np.mean(recalls)) > 0 else 0:.4f}")
    print(f"✅ mAP@0.5: {mAP_50:.4f}")
    print(f"✅ mAP@[0.5:0.95]: {mAP_50_95:.4f}")

# Run the evaluation
calculate_metrics()



✅ Ship Detection Accuracy: 91.09%
✅ Ship Detection Precision: 0.9577
✅ Ship Detection Recall: 0.9587
✅ Ship Detection F1-Score: 0.9582
✅ mAP@0.5: 0.9810
✅ mAP@[0.5:0.95]: 0.8913


In [ ]:
!apt-get install graphviz -y
!pip install graphviz pydot


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 29 not upgraded.


In [ ]:
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt
import cv2

model = unet_model()  # Ensure your model is created

# Save model architecture as an image
plot_model(model, to_file='unet_model.png', show_shapes=True, show_layer_names=True)

# Display the saved architecture in Colab
img = cv2.imread('unet_model.png')
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()


NameError: name 'unet_model' is not defined

In [ ]:
if the plot was not working